# Урок 02 – Ознайомлення з Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** — це єдина платформа для створення AI-агентів. Вона надає чисту, композиційну архітектуру з чотирма основними складовими:

- **Client** – підключається до кінцевої точки AI-моделі та керує комунікацією
- **Agent** – обгортає клієнта із інструкціями та визначеннями інструментів
- **Tools** – розширюють можливості агента за допомогою спеціальних функцій, які модель може викликати
- **Session** – підтримує історію розмови для багатокрокової взаємодії

У цьому уроці ми створимо **агента для бронювання подорожей**, який перевіряє доступність напрямків, використовуючи ці концепції.


## Налаштування


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Розуміння архітектури Agent Framework

Microsoft Agent Framework має шарувату архітектуру:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клієнт** – `FoundryChatClient` підключається до розгортання Azure OpenAI. Він обробляє автентифікацію, форматування запитів і розбір відповідей.
2. **Агент** – Створений із клієнта через `provider.create_agent()`, агент поєднує доступ до моделі з інструкціями (системним підказуванням) та інструментами.
3. **Інструменти** – Функції Python, прикрашені `@tool`, які агент може викликати для виконання дій або отримання даних.
4. **Сесія** – Об’єкт `AgentSession` (створений через `agent.create_session()`), який зберігає історію розмови, що дає змогу вести багатокроковий діалог, де агент пам’ятає попередній контекст.

Давайте розглянемо побудову кожного шару по кроках.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Додавання інструментів за допомогою декоратора @tool

Інструменти дозволяють агентам виконувати дії, окрім генерації тексту. Декоратор `@tool` перетворює звичайну функцію Python на щось, що агент може викликати.

Основні моменти:
- Використовуйте `Annotated[type, "description"]`, щоб модель розуміла кожен параметр.
- Докстрінг стає описом інструмента, який бачить модель.
- `approval_mode="never_require"` означає, що інструмент запускається автоматично без підтвердження користувача.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Створення агента з інструментами

Тепер ми поєднуємо клієнта, інструкції та інструменти в агента. `instructions` діють як системний запит — вони визначають особистість і поведінку агента.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Багатократні розмови з сесіями

`AgentSession` (створена через `agent.create_session()`) відстежує всі повідомлення у розмові. Передаючи однакову сесію кожного разу до виклику `agent.run()`, агент має доступ до повної історії розмови і може звертатися до попередніх повідомлень.

Ми передаємо `tools=[check_destination_availability]`, щоб агент міг викликати наш перевіряльник доступності під час кожного кроку.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Резюме

У цьому уроці ви ознайомилися з чотирма стовпами Microsoft Agent Framework:

| Концепція | Чого ви навчились |
|---------|------------------|
| **Клієнт** | `FoundryChatClient` підключається до Azure OpenAI з автентифікацією на основі облікових даних |
| **Агент** | `provider.create_agent()` об’єднує підключення моделі з інструкціями та назвою |
| **Інструменти** | Декоратор `@tool` відкриває функції Python для виклику агентом |
| **Сесія** | `agent.create_session()` підтримує історію розмови протягом кількох кроків |

Ці будівельні блоки збираються разом, щоб створювати агентів, які можуть вести природні розмови, викликати зовнішні функції та підтримувати контекст — основу для більш просунутих агентських шаблонів у наступних уроках.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
